In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

In [2]:
df = pd.read_csv("../data/processed/df_integrated.csv")
print(f"train loaded: {len(df):,} rows")

train loaded: 46,301,701 rows


In [3]:
# Remove rows where base_fare is 0 or negative
df = df[df["base_fare"] > 0].copy()
print("Rows after removing base_fare <= 0:", len(df))

Rows after removing base_fare <= 0: 44895542


In [4]:
FEATURES = [
    "distance_miles",
    "rider_count",
    "rate_class_id",
    "origin_loc_id",
    "dest_loc_id"
]

TARGET = "base_fare"

df_model = df[FEATURES + [TARGET]].copy()

print(df_model.head())
print(df_model.shape)

   distance_miles  rider_count  rate_class_id  origin_loc_id  dest_loc_id  \
0            0.97          1.0            1.0            239          238   
1            0.90          1.0            1.0            163          162   
2            1.40          1.0            1.0             43          237   
3            5.58          4.0            1.0            142          209   
4            2.16          1.0            1.0             88          144   

   base_fare  
0        7.2  
1        7.9  
2       10.7  
3       38.7  
4       13.5  
(44895542, 6)


In [5]:
X = df_model[FEATURES]
y = df_model[TARGET]

In [6]:
df[["distance_miles", "rider_count", "rate_class_id", 'origin_loc_id', "dest_loc_id", "base_fare"]].corr()

,distance_miles,rider_count,rate_class_id,origin_loc_id,dest_loc_id,base_fare
distance_miles,1.000000,0.021183,0.132590,-0.094994,-0.064814,0.154523
rider_count,0.021183,1.000000,-0.051649,0.008908,0.011829,0.010670
rate_class_id,0.132590,-0.051649,1.000000,-0.085465,-0.075738,0.034140
origin_loc_id,-0.094994,0.008908,-0.085465,1.000000,0.077824,-0.030846
dest_loc_id,-0.064814,0.011829,-0.075738,0.077824,1.000000,-0.021637
base_fare,0.154523,0.010670,0.034140,-0.030846,-0.021637,1.000000


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(35916433, 5)
(8979109, 5)


In [8]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predictions
y_pred = model.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("XGBoost Regression Results")
print("--------------------------")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print(f"R² % : {r2 * 100:.2f}%")

XGBoost Regression Results
--------------------------
MAE  : 3.4225
RMSE : 15.2088
R²   : 0.2663
R² % : 26.63%


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict on training data
y_train_pred = model.predict(X_train)

# Training metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2 = r2_score(y_train, y_train_pred)

print("XGBoost Training Results")
print("------------------------")
print(f"MAE  : {train_mae:.4f}")
print(f"RMSE : {train_rmse:.4f}")
print(f"R²   : {train_r2:.4f}")
print(f"R² % : {train_r2 * 100:.2f}%")

XGBoost Training Results
------------------------
MAE  : 3.4316
RMSE : 59.3132
R²   : 0.4312
R² % : 43.12%


In [13]:
import pickle

with open("xgboost_base_fare_model.pkl", "wb") as file:
    pickle.dump(model, file)

print("Model saved successfully!")

Model saved successfully!
